# Model A: Custom CNN

Binary classification of real and AI-generated images
using a Custom Convolutional Neural Network.

Training environment: Google Colab
Dataset: Defactify Image Dataset
Framework: TensorFlow / Keras

In [ ]:
!pip -q install datasets huggingface_hub

In [ ]:
import os
import json
import shutil
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from PIL import Image as PILImage
from datasets import load_dataset
from huggingface_hub import HfApi

In [ ]:
# GitHub repository
REPO_URL = (
    "https://github.com/Michaelatj/"
    "AI-Generated-Image-Classification-using-CNN.git"
)

REPO_DIR = Path(
    "/content/AI-Generated-Image-Classification-using-CNN"
)

# Clone repository if not available
if not REPO_DIR.exists():

    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )

os.chdir(REPO_DIR)

# Sync with latest GitHub main branch
status = subprocess.check_output(
    ["git", "status", "--porcelain"],
    text=True
)

if status.strip():
    raise RuntimeError(
        "Repository contains local changes. "
        "Check git status before syncing."
    )

subprocess.run(
    ["git", "pull", "--rebase", "origin", "main"],
    check=True
)

# Project directories
ARTIFACT_DIR = REPO_DIR / "artifacts"
SPLIT_DIR = ARTIFACT_DIR / "splits"
CONFIG_DIR = ARTIFACT_DIR / "configs"
MODEL_DIR = ARTIFACT_DIR / "models"
EVAL_DIR = ARTIFACT_DIR / "evaluation"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Repository:", REPO_DIR)
print("Models directory:", MODEL_DIR)

In [ ]:
# TensorFlow configuration
SEED = 42

tf.keras.utils.set_random_seed(SEED)

# GPU configuration
gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True
        )
    except RuntimeError as error:
        print(error)

print("TensorFlow:", tf.__version__)
print("Available GPU:", gpus)

# Check available Colab storage
total, used, free = shutil.disk_usage("/content")

print(f"Free storage: {free / (1024**3):.2f} GB")

## Dataset Loading

Load the prepared CSV manifests and original images
from the Hugging Face Defactify Image Dataset.

Target:
- 0 = Real
- 1 = AI-Generated

The test split is reserved for final evaluation.

In [ ]:
TRAIN_CSV = SPLIT_DIR / "train.csv"
VAL_CSV = SPLIT_DIR / "validation.csv"
TEST_CSV = SPLIT_DIR / "test.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

# Sort every split so image retrieval and prediction order stay consistent
train_df = train_df.sort_values(
    ["source_split", "row_idx"]
).reset_index(drop=True)

val_df = val_df.sort_values(
    ["source_split", "row_idx"]
).reset_index(drop=True)

test_df = test_df.sort_values(
    ["source_split", "row_idx"]
).reset_index(drop=True)

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"{name}: {df.shape}")
    print(
        "  class distribution:",
        df["label_a"].value_counts(normalize=True).round(3).to_dict()
    )

display(train_df.head())

In [ ]:
DATASET_ID = (
    "Rajarshi-Roy-research/Defactify_Image_Dataset"
)

HF_CACHE_DIR = "/content/huggingface_cache"

# Read the pinned dataset revision from the shared manifest.
# Stop instead of silently falling back to the latest revision.
manifest_path = CONFIG_DIR / "dataset_manifest.json"

if not manifest_path.exists():
    raise FileNotFoundError(
        f"{manifest_path} not found. "
        "Pull the latest repository state first."
    )

with open(manifest_path, encoding="utf-8") as f:
    manifest = json.load(f)

HF_REVISION = manifest.get("dataset_revision")

if not HF_REVISION or HF_REVISION == "main":
    raise ValueError(
        "dataset_manifest.json has no pinned dataset_revision."
    )

print("Dataset revision:", HF_REVISION)

HF_DATASETS = {}

# Only load the splits needed for training
required_splits = sorted(
    set(train_df["source_split"]).union(
        set(val_df["source_split"])
    )
)

for split_name in required_splits:

    print(f"\nLoading {split_name}...")

    HF_DATASETS[split_name] = load_dataset(
        DATASET_ID,
        split=split_name,
        revision=HF_REVISION,
        cache_dir=HF_CACHE_DIR
    )

    print(
        f"{split_name}: "
        f"{len(HF_DATASETS[split_name]):,} images"
    )

In [ ]:
# Caption-group leakage check across all three splits
groups = {
    "train": set(train_df["caption_group_id"].dropna()),
    "validation": set(val_df["caption_group_id"].dropna()),
    "test": set(test_df["caption_group_id"].dropna())
}

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    overlap = groups[a] & groups[b]

    print(f"Caption-group overlap {a} - {b}:", len(overlap))

    assert len(overlap) == 0, (
        f"LEAKAGE: {a} and {b} share caption groups"
    )


# Same source row must not appear in two splits
def row_keys(df):
    return set(zip(df["source_split"], df["row_idx"]))


keys = {
    "train": row_keys(train_df),
    "validation": row_keys(val_df),
    "test": row_keys(test_df)
}

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    assert not (keys[a] & keys[b]), (
        f"Same source row in {a} and {b}"
    )

for name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    assert not df[["source_split", "row_idx"]].duplicated().any(), (
        f"Duplicate source rows in {name}"
    )
    assert set(df["label_a"].unique()).issubset({0, 1})

print("\nSplit checks passed: no caption-group or row overlap.")

In [ ]:
def normalize_caption(text):
    return " ".join(
        str(text).strip().split()
    ).casefold()


required_columns = [
    "image_id",
    "source_split",
    "row_idx",
    "caption",
    "label_a",
    "label_b"
]

for name, df in [
    ("train", train_df),
    ("validation", val_df)
]:

    assert set(required_columns).issubset(df.columns)

    assert df["image_id"].is_unique

    assert not df[
        ["source_split", "row_idx", "caption", "label_a", "label_b"]
    ].isnull().any().any()

    assert df[["source_split", "row_idx"]].duplicated().sum() == 0

    assert set(df["label_a"].unique()).issubset({0, 1})


def validate_mapping(df, split_name):

    for source_split, group in df.groupby(
        "source_split",
        sort=False
    ):

        hf_dataset = HF_DATASETS[source_split]

        indices = group["row_idx"].astype(int).tolist()

        if min(indices) < 0 or max(indices) >= len(hf_dataset):
            raise IndexError(
                f"Invalid row_idx in {split_name}"
            )

        # Select original Hugging Face rows
        # without accessing the image column
        selected = hf_dataset.select(indices)

        actual_labels_a = np.asarray(
            selected["Label_A"]
        )

        actual_labels_b = np.asarray(
            selected["Label_B"]
        )

        expected_labels_a = group["label_a"].to_numpy()
        expected_labels_b = group["label_b"].to_numpy()

        assert np.array_equal(
            actual_labels_a,
            expected_labels_a
        ), f"Label_A mismatch: {split_name}"

        assert np.array_equal(
            actual_labels_b,
            expected_labels_b
        ), f"Label_B mismatch: {split_name}"

        actual_captions = [
            normalize_caption(x)
            for x in selected["Caption"]
        ]

        expected_captions = [
            normalize_caption(x)
            for x in group["caption"]
        ]

        assert actual_captions == expected_captions, (
            f"Caption mismatch: {split_name}"
        )

    print(
        f"{split_name}: image mapping verified!"
    )


validate_mapping(train_df, "train")
validate_mapping(val_df, "validation")

# Check caption group leakage
if "caption_group_id" in train_df.columns:

    train_groups = set(train_df["caption_group_id"].dropna())
    val_groups = set(val_df["caption_group_id"].dropna())

    overlap = train_groups & val_groups

    assert len(overlap) == 0, (
        f"Caption group leakage detected: {len(overlap)} groups"
    )

    print("No caption group leakage detected.")

In [ ]:
fig, axes = plt.subplots(
    2, 4,
    figsize=(14, 7)
)

sample_rows = train_df.sample(
    n=8,
    random_state=SEED
)

for ax, (_, row) in zip(
    axes.flatten(),
    sample_rows.iterrows()
):

    sample = HF_DATASETS[
        row["source_split"]
    ][int(row["row_idx"])]

    image = sample["Image"].convert("RGB")

    ax.imshow(image)

    label_name = (
        "Real"
        if row["label_a"] == 0
        else "AI-Generated"
    )

    ax.set_title(label_name)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Image Preprocessing and Data Pipeline

Retrieve images using source_split and row_idx.

Preprocessing:
- RGB conversion.
- Resize to 224 × 224.
- Convert to uint8 image tensors.
- Normalize pixel values inside the CNN.

Use tf.data for batching, shuffling, and prefetching.

In [ ]:
IMAGE_SIZE = 224
IMAGE_SHAPE = (224, 224, 3)

BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Calculate class weights from training data
class_counts = (
    train_df["label_a"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)

if (class_counts == 0).any():
    raise ValueError("A training class is missing!")

total_samples = len(train_df)

class_weights = {
    label: total_samples / (
        2 * class_counts[label]
    )
    for label in [0, 1]
}

print("Class distribution:")
print(class_counts)

print("\nClass weights:")
print(class_weights)


def image_generator(df, training=False, rng=None):

    # Full permutation of the training rows on every pass.
    # This replaces the small tf.data shuffle buffer.
    if training and rng is not None:
        order = rng.permutation(len(df))
        df = df.iloc[order].reset_index(drop=True)

    for row in df.itertuples(index=False):

        # Retrieve original image
        sample = HF_DATASETS[
            row.source_split
        ][int(row.row_idx)]

        image = sample["Image"]

        if image is None:
            raise ValueError(
                f"Missing image: {row.image_id}"
            )

        # Convert to RGB and resize
        image = image.convert("RGB")

        image = image.resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            resample=PILImage.Resampling.BILINEAR
        )

        # Convert to NumPy array
        image_array = np.asarray(
            image,
            dtype=np.uint8
        )

        # Binary label
        label = np.array(
            [int(row.label_a)],
            dtype=np.float32
        )

        # Apply class weighting only to training data
        sample_weight = np.float32(
            class_weights[int(row.label_a)]
            if training else 1.0
        )

        yield image_array, label, sample_weight

In [ ]:
OUTPUT_SIGNATURE = (

    tf.TensorSpec(
        shape=IMAGE_SHAPE,
        dtype=tf.uint8
    ),

    tf.TensorSpec(
        shape=(1,),
        dtype=tf.float32
    ),

    tf.TensorSpec(
        shape=(),
        dtype=tf.float32
    )
)


def create_tf_dataset(df, training=False, shuffle_seed=SEED):

    # One generator per dataset, so every dataset has its own
    # reproducible shuffle sequence (tuning runs do not affect it)
    rng = (
        np.random.default_rng(shuffle_seed)
        if training else None
    )

    dataset = tf.data.Dataset.from_generator(
        lambda: image_generator(
            df,
            training=training,
            rng=rng
        ),
        output_signature=OUTPUT_SIGNATURE
    )

    # Verify expected number of samples
    dataset = dataset.apply(
        tf.data.experimental.assert_cardinality(
            len(df)
        )
    )

    dataset = dataset.batch(
        BATCH_SIZE
    )

    dataset = dataset.prefetch(
        AUTOTUNE
    )

    return dataset


train_dataset = create_tf_dataset(
    train_df,
    training=True
)

val_dataset = create_tf_dataset(
    val_df,
    training=False
)

print("Train and validation pipelines created!")

In [ ]:
images, labels, sample_weights = next(
    iter(train_dataset)
)

print("Image shape:", images.shape)
print("Label shape:", labels.shape)
print("Sample weight shape:", sample_weights.shape)

print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

# Display training samples
plt.figure(figsize=(12, 7))

for i in range(min(8, len(images))):

    plt.subplot(2, 4, i + 1)

    plt.imshow(
        images[i].numpy()
    )

    label = int(
        labels[i][0].numpy()
    )

    plt.title(
        "Real" if label == 0 else "AI-Generated"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

## Custom CNN Architecture

Build a CNN from scratch for binary image classification.

Architecture:
- Input: 224 × 224 × 3
- Three convolution blocks: 32, 64, 128 filters
- Batch Normalization
- ReLU activation
- Max Pooling
- Global Average Pooling
- Dense: 128 neurons
- Dropout (rate selected by hyperparameter tuning)
- Sigmoid output

Training configuration:
- Optimizer: Adam
- Learning rate: selected by hyperparameter tuning
- Loss: Binary Crossentropy
- Batch size: 32
- Maximum epochs: 10

In [ ]:
from tensorflow.keras import layers, models


def conv_block(x, filters, dropout_rate=0.0):

    # First convolution
    x = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # Second convolution
    x = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # Downsampling
    x = layers.MaxPooling2D(
        pool_size=2
    )(x)

    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)

    return x


def build_custom_cnn(head_dropout=0.4):

    inputs = layers.Input(
        shape=IMAGE_SHAPE
    )

    # Normalize image pixels from 0-255 to 0-1
    x = layers.Rescaling(1.0 / 255)(inputs)

    # Convolution blocks
    x = conv_block(x, 32)
    x = conv_block(x, 64, dropout_rate=0.10)
    x = conv_block(x, 128, dropout_rate=0.20)

    # Feature aggregation
    x = layers.GlobalAveragePooling2D()(x)

    # Classification head
    x = layers.Dense(
        128,
        activation="relu"
    )(x)

    x = layers.Dropout(head_dropout)(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    return models.Model(
        inputs=inputs,
        outputs=outputs,
        name="Custom_CNN"
    )


model = build_custom_cnn()

model.summary()

In [ ]:
def compile_model(model, learning_rate):

    model.compile(

        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),

        loss="binary_crossentropy",

        metrics=[

            tf.keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),

            tf.keras.metrics.Precision(
                name="precision"
            ),

            tf.keras.metrics.Recall(
                name="recall"
            ),

            tf.keras.metrics.AUC(
                name="auc"
            )
        ]
    )

    return model


compile_model(model, 1e-3)

print("Custom CNN compiled successfully!")

In [ ]:
RUN_TUNING = True
TUNE_TRAIN_SIZE = 6000
TUNE_VAL_SIZE = 1500
TUNE_EPOCHS = 2

DEFAULT_CONFIG = {
    "learning_rate": 1e-3,
    "head_dropout": 0.4
}

tuning_grid = [
    {"learning_rate": 1e-3, "head_dropout": 0.3},
    {"learning_rate": 1e-3, "head_dropout": 0.5},
    {"learning_rate": 3e-4, "head_dropout": 0.3},
    {"learning_rate": 3e-4, "head_dropout": 0.5}
]

tuning_df = pd.DataFrame()
best_config = dict(DEFAULT_CONFIG)

if RUN_TUNING:

    # Small random subsets, same procedure as Model B
    tune_train = (
        train_df
        .sample(
            n=min(TUNE_TRAIN_SIZE, len(train_df)),
            random_state=SEED
        )
        .sort_values(["source_split", "row_idx"])
        .reset_index(drop=True)
    )

    tune_val = (
        val_df
        .sample(
            n=min(TUNE_VAL_SIZE, len(val_df)),
            random_state=SEED
        )
        .sort_values(["source_split", "row_idx"])
        .reset_index(drop=True)
    )

    tune_train_ds = create_tf_dataset(tune_train, training=True)
    tune_val_ds = create_tf_dataset(tune_val, training=False)

    rows = []

    for cfg in tuning_grid:

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)

        m = compile_model(
            build_custom_cnn(cfg["head_dropout"]),
            cfg["learning_rate"]
        )

        h = m.fit(
            tune_train_ds,
            validation_data=tune_val_ds,
            epochs=TUNE_EPOCHS,
            verbose=0
        )

        best_epoch = int(np.argmin(h.history["val_loss"]))

        rows.append({
            **cfg,
            "best_val_loss": h.history["val_loss"][best_epoch],
            "val_accuracy": h.history["val_accuracy"][best_epoch],
            "val_auc": h.history["val_auc"][best_epoch]
        })

        print(rows[-1])

    tuning_df = (
        pd.DataFrame(rows)
        .sort_values("best_val_loss")
        .reset_index(drop=True)
    )

    display(tuning_df)

    best_config = {
        "learning_rate": float(tuning_df.loc[0, "learning_rate"]),
        "head_dropout": float(tuning_df.loc[0, "head_dropout"])
    }

print("Selected config:", best_config)

In [ ]:
BEST_MODEL_PATH = (
    MODEL_DIR / "custom_cnn_best.keras"
)

callbacks = [

    # Save the model with the best validation loss
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    # Stop training when validation performance stops improving
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    # Reduce learning rate when validation loss stagnates
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
EPOCHS = 10

# Rebuild from scratch with the selected configuration
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

model = build_custom_cnn(
    head_dropout=best_config["head_dropout"]
)

compile_model(
    model,
    best_config["learning_rate"]
)

history = model.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=EPOCHS,

    callbacks=callbacks,

    verbose=1
)

print("Training completed!")

## Model Evaluation and Artifact Storage

Evaluate the best Custom CNN using validation data.

Save:
- custom_cnn_best.keras
- custom_cnn_history.csv
- custom_cnn_config.json
- custom_cnn_tuning.csv
- evaluation metrics, classification report, confusion matrix, and training curves

Push generated artifacts to GitHub.
The test split remains reserved for final model comparison.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)

# Standard Keras metrics (threshold 0.5)
keras_metrics = best_model.evaluate(
    val_dataset,
    return_dict=True,
    verbose=0
)

# Predicted probabilities, in the same order as val_df
y_true = val_df["label_a"].to_numpy().astype(int)

val_images_only = val_dataset.map(
    lambda image, label, weight: image
)

y_prob = best_model.predict(
    val_images_only,
    verbose=1
).ravel()

assert len(y_prob) == len(y_true)

# Pick the threshold that maximizes macro-F1 on validation
thresholds = np.arange(0.05, 0.96, 0.01)

macro_f1_scores = [
    f1_score(
        y_true,
        (y_prob >= t).astype(int),
        average="macro",
        zero_division=0
    )
    for t in thresholds
]

best_threshold = float(
    thresholds[int(np.argmax(macro_f1_scores))]
)


def summarize(threshold):

    y_pred = (y_prob >= threshold).astype(int)

    return {
        "threshold": round(float(threshold), 2),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "precision_ai": float(
            precision_score(y_true, y_pred, zero_division=0)
        ),
        "recall_ai": float(
            recall_score(y_true, y_pred, zero_division=0)
        ),
        "recall_real": float(
            recall_score(y_true, y_pred, pos_label=0, zero_division=0)
        ),
        "f1_ai": float(f1_score(y_true, y_pred, zero_division=0)),
        "f1_macro": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(
            y_true, y_pred, labels=[0, 1]
        ).tolist()
    }


val_metrics = {
    "keras_evaluate": {
        k: float(v) for k, v in keras_metrics.items()
    },
    "roc_auc": float(roc_auc_score(y_true, y_prob)),
    "at_threshold_0.50": summarize(0.5),
    "at_tuned_threshold": summarize(best_threshold),
    "tuned_threshold": round(best_threshold, 2),
    "threshold_selection": "max macro-F1 on validation set"
}

print("VALIDATION RESULTS")
print("ROC-AUC:", round(val_metrics["roc_auc"], 4))

for key in ["at_threshold_0.50", "at_tuned_threshold"]:

    r = val_metrics[key]

    print(
        f"\n{key} (threshold={r['threshold']})\n"
        f"  accuracy={r['accuracy']:.4f}  "
        f"balanced_acc={r['balanced_accuracy']:.4f}  "
        f"macro_f1={r['f1_macro']:.4f}\n"
        f"  recall_real={r['recall_real']:.4f}  "
        f"recall_ai={r['recall_ai']:.4f}  "
        f"precision_ai={r['precision_ai']:.4f}"
    )

# The tuned threshold is chosen on this same validation set,
# so its numbers are slightly optimistic. Freeze the threshold
# and report the final numbers on the test split only.
y_pred_tuned = (y_prob >= best_threshold).astype(int)

report_df = pd.DataFrame(
    classification_report(
        y_true,
        y_pred_tuned,
        target_names=["Real", "AI-Generated"],
        output_dict=True,
        zero_division=0
    )
).transpose()

display(report_df)

# Confusion matrix figure
cm = np.array(val_metrics["at_tuned_threshold"]["confusion_matrix"])

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Real", "AI-Generated"])
ax.set_yticklabels(["Real", "AI-Generated"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Validation confusion matrix (threshold={best_threshold:.2f})")

for i in range(2):
    for j in range(2):
        ax.text(
            j, i, f"{cm[i, j]:,}",
            ha="center", va="center",
            color="white" if cm[i, j] > cm.max() / 2 else "black"
        )

plt.tight_layout()
plt.savefig(
    EVAL_DIR / "custom_cnn_confusion_matrix.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

# Save metrics and report
METRICS_PATH = EVAL_DIR / "custom_cnn_metrics.json"
REPORT_PATH = EVAL_DIR / "custom_cnn_classification_report.csv"

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(val_metrics, f, indent=4)

report_df.to_csv(REPORT_PATH)

In [ ]:
history_df = pd.DataFrame(history.history)

# Save training history
HISTORY_PATH = (
    MODEL_DIR / "custom_cnn_history.csv"
)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)

# Plot accuracy and loss
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

axes[0].plot(
    history_df["accuracy"],
    label="Training Accuracy"
)

axes[0].plot(
    history_df["val_accuracy"],
    label="Validation Accuracy"
)

axes[0].set_title("Training vs Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)


axes[1].plot(
    history_df["loss"],
    label="Training Loss (sample-weighted)"
)

axes[1].plot(
    history_df["val_loss"],
    label="Validation Loss"
)

axes[1].set_title("Training vs Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(
    EVAL_DIR / "custom_cnn_training_curves.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

display(history_df)

In [ ]:
model_size_mb = (
    BEST_MODEL_PATH.stat().st_size / (1024 ** 2)
)

training_config = {

    "experiment": "Model A - Custom CNN",

    "dataset": DATASET_ID,

    "dataset_revision": HF_REVISION,

    "target": "label_a",

    "class_mapping": {
        "0": "Real",
        "1": "AI-Generated"
    },

    "image_size": IMAGE_SIZE,

    "color_mode": "RGB",

    "resize_interpolation": "bilinear",

    "model_specific_preprocessing": (
        "Rescaling(1/255) inside the model. "
        "The saved model expects raw 0-255 RGB, 224x224."
    ),

    "augmentation_training_only": "none",

    "shuffle": "full permutation of training rows every epoch",

    "architecture": (
        "Custom CNN, 3 convolution blocks (32, 64, 128 filters, "
        "2 Conv-BN-ReLU per block, MaxPooling), "
        "GlobalAveragePooling2D, Dense(128), Dropout, Dense(1, sigmoid)"
    ),

    "head_dropout": best_config["head_dropout"],

    "batch_size": BATCH_SIZE,

    "optimizer": "Adam",

    "learning_rate": best_config["learning_rate"],

    "loss": "Binary Crossentropy",

    "epochs_requested": EPOCHS,

    "epochs_completed": len(history.epoch),

    "class_weights": {
        str(k): float(v)
        for k, v in class_weights.items()
    },

    "seed": SEED,

    "tuning": {
        "ran": RUN_TUNING,
        "subset_train": TUNE_TRAIN_SIZE,
        "subset_val": TUNE_VAL_SIZE,
        "epochs": TUNE_EPOCHS,
        "results": tuning_df.to_dict(orient="records"),
        "selected": best_config
    },

    "validation_results": val_metrics,

    "decision_threshold": val_metrics["tuned_threshold"],

    "decision_threshold_selection": (
        "max macro-F1 on validation set"
    ),

    "training_samples": len(train_df),

    "validation_samples": len(val_df),

    "model_file": BEST_MODEL_PATH.name,

    "model_size_mb": round(model_size_mb, 2),

    "total_parameters": int(best_model.count_params())
}

CONFIG_PATH = (
    MODEL_DIR / "custom_cnn_config.json"
)

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        training_config,
        f,
        indent=4
    )

TUNING_PATH = MODEL_DIR / "custom_cnn_tuning.csv"

if len(tuning_df):
    tuning_df.to_csv(TUNING_PATH, index=False)

print("Model and experiment configuration saved!")

for path in [
    BEST_MODEL_PATH,
    HISTORY_PATH,
    CONFIG_PATH,
    TUNING_PATH,
    METRICS_PATH,
    REPORT_PATH
]:

    if path.exists():
        print(
            path.name,
            f"{path.stat().st_size / (1024**2):.2f} MB"
        )

In [ ]:
import base64
from google.colab import userdata

# Retrieve GitHub token from Colab Secrets
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise ValueError("GITHUB_TOKEN is missing!")

# Files to upload. The first three are required,
# the rest are pushed only if they exist.
required_files = [
    BEST_MODEL_PATH,
    HISTORY_PATH,
    CONFIG_PATH
]

optional_files = [
    TUNING_PATH,
    METRICS_PATH,
    REPORT_PATH,
    EVAL_DIR / "custom_cnn_training_curves.png",
    EVAL_DIR / "custom_cnn_confusion_matrix.png"
]

for p in required_files:
    if not p.exists():
        raise FileNotFoundError(p)

files_to_push = required_files + [
    p for p in optional_files if p.exists()
]

# Verify files and Git ignore status
for file_path in files_to_push:

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    if file_path.stat().st_size >= 100 * 1024 * 1024:
        raise ValueError(
            f"{file_path.name} exceeds GitHub's "
            "100 MB single-file limit. Use Git LFS."
        )

    relative_path = str(
        file_path.relative_to(REPO_DIR)
    )

    ignored = subprocess.run(
        ["git", "check-ignore", "-q", relative_path],
        cwd=REPO_DIR
    )

    if ignored.returncode == 0:
        raise RuntimeError(
            f"{relative_path} is ignored by .gitignore"
        )

# Configure Git identity
subprocess.run(
    [
        "git", "config", "--local",
        "user.name", "Michael Andreas Tjendra"
    ],
    cwd=REPO_DIR,
    check=True
)

subprocess.run(
    [
        "git", "config", "--local",
        "user.email", "alongbest2000@gmail.com"
    ],
    cwd=REPO_DIR,
    check=True
)

# Stage model artifacts
subprocess.run(
    [
        "git", "add", "--"
    ] + [
        str(p.relative_to(REPO_DIR))
        for p in files_to_push
    ],
    cwd=REPO_DIR,
    check=True
)

# Commit staged files if there are changes
diff_result = subprocess.run(
    ["git", "diff", "--cached", "--quiet"],
    cwd=REPO_DIR
)

if diff_result.returncode != 0:

    subprocess.run(
        [
            "git", "commit",
            "-m", "Train and save Custom CNN model"
        ],
        cwd=REPO_DIR,
        check=True
    )

else:
    print("No new changes to commit.")

# Sync remote before pushing
subprocess.run(
    ["git", "fetch", "origin"],
    cwd=REPO_DIR,
    check=True
)

subprocess.run(
    ["git", "pull", "--rebase", "origin", "main"],
    cwd=REPO_DIR,
    check=True
)

# Temporary Git authentication
askpass_path = Path("/tmp/github-askpass.sh")

askpass_path.write_text("""#!/bin/sh
case "$1" in
    *sername*) printf '%s\\n' 'Michaelatj' ;;
    *assword*) printf '%s\\n' "$GITHUB_TOKEN" ;;
    *) exit 1 ;;
esac
""")

os.chmod(askpass_path, 0o700)

env = os.environ.copy()

env["GIT_ASKPASS"] = str(askpass_path)
env["GIT_TERMINAL_PROMPT"] = "0"
env["GITHUB_TOKEN"] = token

# Push
result = subprocess.run(
    ["git", "push", "origin", "main"],
    cwd=REPO_DIR,
    env=env,
    capture_output=True,
    text=True
)

if result.returncode != 0:

    print(result.stderr)

    raise RuntimeError(
        "GitHub push failed. Check repository synchronization."
    )

print(result.stdout)
print("Custom CNN artifacts successfully pushed!")

In [ ]:
subprocess.run(
    [
        "git",
        "status",
        "--short",
        "--branch"
    ],
    cwd=REPO_DIR,
    check=True
)